# Independent Python audit

This notebook is an **independent audit layer** for the Figshare source package. It does not read numbers from the associated paper and does not replace the R analysis. Its job is to verify the raw Excel matrix, reconstruct the documented scales, calculate selected headline statistics independently, and reconcile the Excel source with the SPSS `.sav` file.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "python"))
from audit_utils import read_xlsx, validate_raw, score_constructs, core_statistics, compare_sav, cronbach_alpha, CONSTRUCTS
XLSX = ROOT / "data/original/Dataset_S1_Raw_Survey_Data.xlsx"
SAV = ROOT / "data/original/Dataset_S2_SPSS_Data_File.sav"

In [ ]:
raw, codebook = read_xlsx(XLSX)
validate_raw(raw)
print("raw shape:", raw.shape)
print("missing cells:", int(raw.isna().sum().sum()))
print("response range:", int(raw.min().min()), "to", int(raw.max().max()))
codebook.head(10)

In [ ]:
scores = score_constructs(raw)
scores.describe().T

In [ ]:
import pandas as pd
reliability = pd.DataFrame([
    {"construct": k, "items": len(cols), "cronbach_alpha": cronbach_alpha(raw[cols])}
    for k, cols in CONSTRUCTS.items()
])
reliability

In [ ]:
stats = core_statistics(raw)
stats[stats.metric.isin(["pearson_r", "pearson_p", "spearman_rho", "ols_standardized_beta", "model_r_squared", "model_adjusted_r_squared"])]

In [ ]:
check = compare_sav(raw, scores, SAV)
print(json.dumps(check, indent=2))
if check.get("available"):
    assert check["raw_max_abs_difference"] == 0
    assert all(v < 1e-10 for v in check["composite_max_abs_difference"].values())
    print("XLSX ↔ SPSS SAV reconciliation: PASS")
else:
    print("SAV reconciliation deferred: pyreadstat is unavailable in this local environment. GitHub Actions installs it.")

## Interpretation boundary

These calculations are derived from the supplied Figshare files. They do not use manuscript-reported numerical targets, do not alter the paper, and do not imply causality or population representativeness.